In [ ]:
import random
import pickle
from tensorflow.keras.models import load_model
import numpy as np

model = load_model("../models/midi_lstm_model.h5")

with open("../models/int_to_note.pkl", "rb") as f:
    int_to_note = pickle.load(f)

start = random.randint(0, len(X_midi)-1)
pattern = list(X_midi[start].flatten())

generated_notes = []

for _ in range(300):
    input_seq = np.reshape(pattern, (1, len(pattern), 1))
    prediction = model.predict(input_seq, verbose=0)
    index = np.argmax(prediction)
    result = int_to_note[index]

    generated_notes.append(result)
    pattern.append(index / float(len(int_to_note)))
    pattern = pattern[1:]


In [ ]:
from music21 import stream, note, chord

output_notes = []

for pattern in generated_notes:
    if "." in pattern:
        notes = pattern.split(".")
        chord_notes = [note.Note(int(n)) for n in notes]
        output_notes.append(chord.Chord(chord_notes))
    else:
        output_notes.append(note.Note(pattern))

midi_stream = stream.Stream(output_notes)
midi_stream.write("midi", "../outputs/generated_music.mid")

print("Music generated!")
